In [1]:
from __future__ import annotations
import re
import os
from dataclasses import dataclass
from pathlib import Path
import glob
import numpy as np
import xarray as xr
import cftime

from ExperimentUtil import (
    run_dict,
    open_var_from_files,
    get_comp_dir,
    find_nc_files,
    collect_variable_list,
    flatten_variable_dict,
    open_compvar,
    find_files_for_compvar,
    set_monthly_time_to_midmonth_cftime
)


In [2]:
# -------------------------
# Config
# -------------------------
@dataclass
class SswMonthlyConfig:
    # variable names in files
    ua_var: str = "ua"
    va_var: str = "va"
    ta_var: str = "ta"
    zg_var: str = "zg"

    # dimension names (match your var_cfg["dims"])
    time_name: str = "time"
    p_name: str = "plev"
    lat_name: str = "lat"
    lon_name: str = "lon"

    # pressure levels to keep (Pa)
    target_plev_pa: np.ndarray | None = None

    # vortex / polar diagnostics
    polar_cap: tuple[float, float] = (60.0, 90.0)     # 60–90N
    u_lat: float = 60.0                                # 60N line
    u_lat_band: tuple[float, float] | None = (55.0, 65.0)  # if you prefer band mean
    vortex_lat_band: tuple[float, float] = (55.0, 75.0)    # robust alt metric

    # time subsets for RMSE summaries
    # month numbers: 1..12
    winter_months: tuple[int, ...] = (12, 1, 2)        # DJF

def open_var_from_files(files, varname):
    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        decode_times=False,        # keep simple; we'll set time ourselves
        data_vars="minimal",       # avoids FutureWarning behavior changes
        coords="minimal",
        compat="override",
    )
    if varname not in ds:
        raise KeyError(f"Variable '{varname}' not found in files: {files}")
    return ds[varname]
    
def monthly_time_from_filenames_midmonth(files, calendar_cls=cftime.DatetimeNoLeap):
    """
    Build a monthly mid-month time axis by parsing YYYYMM_YYYYMM from filenames like:
      ua_201201_201212.nc, ta_201301_201312.nc
    Returns a list of cftime objects: YYYY-MM-16 00:00 for each month in order of files.
    """
    times = []
    pat = re.compile(r".*_(\d{6})_(\d{6})\.nc$")  # ..._YYYYMM_YYYYMM.nc

    for f in files:
        m = pat.match(os.path.basename(f))
        if not m:
            raise ValueError(f"Cannot parse YYYYMM range from filename: {f}")
        yyyymm0 = m.group(1)
        yyyymm1 = m.group(2)
        y0, m0 = int(yyyymm0[:4]), int(yyyymm0[4:])
        y1, m1 = int(yyyymm1[:4]), int(yyyymm1[4:])

        # iterate months inclusive
        y, mo = y0, m0
        while (y < y1) or (y == y1 and mo <= m1):
            times.append(calendar_cls(y, mo, 16))
            mo += 1
            if mo == 13:
                mo = 1
                y += 1

    return times


def ensure_plev_pa(da: xr.DataArray, p_name="plev") -> xr.DataArray:
    # rename lev->plev if needed
    if "lev" in da.dims and p_name not in da.dims:
        da = da.rename({"lev": p_name})

    if p_name not in da.coords:
        return da

    units = (da[p_name].attrs.get("units") or "").strip().lower()
    vals = da[p_name].values.astype(np.float64)

    if units in ("hpa", "mb", "mbar", "millibar", "millibars"):
        da = da.assign_coords({p_name: vals * 100.0})
        da[p_name].attrs["units"] = "Pa"
    elif units in ("pa", "pascal", "pascals", ""):
        # infer if missing
        if units == "":
            if float(np.nanmax(vals)) < 2000:  # likely hPa
                da = da.assign_coords({p_name: vals * 100.0})
            da[p_name].attrs["units"] = "Pa"
        else:
            da[p_name].attrs["units"] = "Pa"
    else:
        raise ValueError(f"Unrecognized {p_name} units: '{da[p_name].attrs.get('units')}'")
    return da

def load_reanalysis_monthly(ref_dict, var, *, sub="post/atm/180x360_aave/ts/monthly/1yr", p_name="plev"):
    """
    Load monthly reanalysis saved in the same 'post/atm/180x360_aave/ts/monthly/1yr' layout,
    with filenames like 'ua_*.nc', 'ta_*.nc', 'zg_*.nc'.

    Returns DataArray with time set to mid-month cftime.NoLeap (consistent with model).
    """
    base = Path(ref_dict["path"]) / sub
    files = sorted(glob.glob(str(base / f"{var}_*.nc")))
    if not files:
        raise FileNotFoundError(f"No reanalysis files matched: {base / f'{var}_*.nc'}")

    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        data_vars="minimal",
        coords="minimal",
        compat="override",
    )
    if var not in ds:
        raise KeyError(f"Variable '{var}' not found in {files[:2]}... vars={list(ds.data_vars)[:20]}")

    da = ds[var]

    # rename lev->plev if needed
    if "lev" in da.dims and p_name not in da.dims:
        da = da.rename({"lev": p_name})

    # normalize time to mid-month (works for datetime64 or cftime)
    da = set_monthly_time_to_midmonth_cftime(da, time_name="time", day=16, calendar_cls=cftime.DatetimeNoLeap)
    for k in ("calendar", "units"):
        da["time"].attrs.pop(k, None)

    # ensure pressure is Pa (reuse your existing helper logic if you already have one)
    if p_name in da.coords:
        units = (da[p_name].attrs.get("units") or "").strip().lower()
        vals = da[p_name].values.astype(np.float64)
        if units in ("hpa", "mb", "mbar", "millibar", "millibars"):
            da = da.assign_coords({p_name: vals * 100.0})
            da[p_name].attrs["units"] = "Pa"
        elif units in ("pa", "pascal", "pascals", ""):
            if units == "" and float(np.nanmax(vals)) < 2000:
                da = da.assign_coords({p_name: vals * 100.0})
            da[p_name].attrs["units"] = "Pa"

    return da.sortby("time")

def load_era5_native_monthly(ref_dict, var_native, *, sub="post/atm/180x360_aave/ts/monthly/1yr"):
    """
    Load native-style ERA5 files like:
      U_201201_201212.nc, T_201201_201212.nc, Z3_201201_201212.nc
    """
    base = Path(ref_dict["path"]) / sub
    files = sorted(glob.glob(str(base / f"{var_native}_*.nc")))
    if not files:
        raise FileNotFoundError(f"No ERA5 files matched: {base / f'{var_native}_*.nc'}")

    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        data_vars="minimal",
        coords="minimal",
        compat="override",
    )
    if var_native not in ds:
        raise KeyError(f"Variable '{var_native}' not in dataset. vars={list(ds.data_vars)[:30]}")

    da = ds[var_native]

    # normalize time to mid-month (your utility handles datetime64 and cftime)
    da = set_monthly_time_to_midmonth_cftime(
        da, time_name="time", day=16, calendar_cls=cftime.DatetimeNoLeap
    )
    for k in ("calendar", "units"):
        da["time"].attrs.pop(k, None)

    # standardize pressure dimension name to plev (if needed)
    if "lev" in da.dims and "plev" not in da.dims:
        da = da.rename({"lev": "plev"})

    # ensure pressure in Pa
    if "plev" in da.coords:
        units = (da["plev"].attrs.get("units") or "").strip().lower()
        vals = da["plev"].values.astype(np.float64)
        if units in ("hpa", "mb", "mbar", "millibar", "millibars"):
            da = da.assign_coords({"plev": vals * 100.0})
            da["plev"].attrs["units"] = "Pa"
        elif units in ("pa", "pascal", "pascals", ""):
            if units == "" and float(np.nanmax(vals)) < 2000:
                da = da.assign_coords({"plev": vals * 100.0})
            da["plev"].attrs["units"] = "Pa"

    return da.sortby("time")


def load_era5_for_ssw(ref_dict):
    """
    Return ERA5 monthly ua/ta/zg DataArrays compatible with SswMonthlyAnalyzer.
    """
    da_u  = load_era5_native_monthly(ref_dict, "U").rename("ua")
    da_t  = load_era5_native_monthly(ref_dict, "T").rename("ta")
    da_z3 = load_era5_native_monthly(ref_dict, "Z3").rename("zg")
    return da_u, da_t, da_z3
    
def load_eam_cmip6_monthly(run_key: str, var: str) -> xr.DataArray:
    run = run_dict[run_key]
    files = find_files_for_compvar(run, "EAM_CMIP6", var)   # <- cmip_ts path
    da = open_var_from_files(files, var)

    # set mid-month time (your utility already handles datetime64 + cftime)
    da = set_monthly_time_to_midmonth_cftime(
        da, time_name="time", day=16, calendar_cls=cftime.DatetimeNoLeap
    )
    for k in ("calendar", "units"):
        da["time"].attrs.pop(k, None)

    da = ensure_plev_pa(da, p_name="plev")
    da = da.sortby("time")

    return da
        
# -------------------------
# Analyzer
# -------------------------
class SswMonthlyAnalyzer:
    """
    Monthly 'SSW-relevant' vortex diagnostics using only monthly mean ua/ta/zg (and optionally va).

    Outputs:
      - monthly time series indices: u60(p), zpc(p), tpc(p), uarea_vortex(p)
      - if reference indices provided: bias time series and RMSE summaries
    """

    def __init__(self, cfg: SswMonthlyConfig):
        self.cfg = cfg

    # ---------- helpers ----------
    def _require_dims(self, da: xr.DataArray, need: tuple[str, ...], name: str):
        missing = set(need) - set(da.dims)
        if missing:
            raise ValueError(f"{name}: missing dims {missing}. got dims={da.dims}")

    def _coslat_weights(self, lat: xr.DataArray) -> xr.DataArray:
        w = np.cos(np.deg2rad(lat))
        return xr.DataArray(w, dims=(self.cfg.lat_name,), coords={self.cfg.lat_name: lat})

    def _sel_lat_band(self, da: xr.DataArray, lat_band: tuple[float, float]) -> xr.DataArray:
        lat0, lat1 = lat_band
        # ensure ascending slice works regardless of coordinate order
        lat = da[self.cfg.lat_name]
        if float(lat[0]) < float(lat[-1]):
            return da.sel({self.cfg.lat_name: slice(lat0, lat1)})
        else:
            return da.sel({self.cfg.lat_name: slice(lat1, lat0)})

    def _zonal_mean(self, da: xr.DataArray) -> xr.DataArray:
        if self.cfg.lon_name in da.dims:
            return da.mean(self.cfg.lon_name)
        return da

    def _area_mean_latlon(self, da: xr.DataArray, lat_band: tuple[float, float]) -> xr.DataArray:
        da2 = self._sel_lat_band(da, lat_band)
        da2 = self._zonal_mean(da2)
        w = self._coslat_weights(da2[self.cfg.lat_name])
        return da2.weighted(w).mean(self.cfg.lat_name)

    def _pick_plev(self, da: xr.DataArray) -> xr.DataArray:
        if self.cfg.target_plev_pa is None:
            return da
        p = self.cfg.p_name
        if p not in da.dims:
            raise ValueError(f"Pressure dim '{p}' not found in da.dims={da.dims}")
        # assume da already has p in Pa and matches; if not, do your existing unit conversion upstream
        target = xr.DataArray(self.cfg.target_plev_pa, dims="plev_target")
        # try exact first, then nearest with small tolerance
        lev_vals = da[p].values.astype(np.float64)
        exact = np.isin(self.cfg.target_plev_pa, lev_vals)
        if exact.all():
            out = da.sel({p: self.cfg.target_plev_pa})
        else:
            tol = 1.0  # Pa
            out = da.sel({p: target}, method="nearest", tolerance=tol)
            out = out.rename({"plev_target": p}).assign_coords({p: self.cfg.target_plev_pa})
            if out.sizes[p] != len(self.cfg.target_plev_pa):
                raise ValueError(
                    f"Nearest p selection failed: got {out.sizes[p]} levels, expected {len(self.cfg.target_plev_pa)}. "
                    f"Increase tolerance."
                )
        return out.sel({p: self.cfg.target_plev_pa})

    def _month_numbers(self, time: xr.DataArray) -> xr.DataArray:
        # Works with cftime or numpy datetime64
        vals = time.values
        months = np.array([getattr(t, "month", int(str(t)[5:7])) for t in vals], dtype=int)
        return xr.DataArray(months, dims=(self.cfg.time_name,), coords={self.cfg.time_name: time})

    # ---------- core computations ----------
    def compute_indices(
        self,
        ua: xr.DataArray,
        ta: xr.DataArray,
        zg: xr.DataArray,
        name: str = "EXP",
    ) -> xr.Dataset:
        c = self.cfg
        need = (c.time_name, c.p_name, c.lat_name)

        self._require_dims(ua, need, f"{name}.ua")
        self._require_dims(ta, need, f"{name}.ta")
        self._require_dims(zg, need, f"{name}.zg")

        ua = self._pick_plev(ua)
        ta = self._pick_plev(ta)
        zg = self._pick_plev(zg)

        # Zonal means first (SSW definitions are zonal-mean based)
        ua_zm = self._zonal_mean(ua)
        ta_zm = self._zonal_mean(ta)
        zg_zm = self._zonal_mean(zg)

        # u60 index: either line at 60N or band mean 55–65N
        if c.u_lat_band is not None:
            u60 = self._area_mean_latlon(ua_zm, c.u_lat_band)  # now dims: time, plev
            u60_name = f"u_{c.u_lat_band[0]:.0f}_{c.u_lat_band[1]:.0f}N"
        else:
            # nearest latitude
            u60 = ua_zm.sel({c.lat_name: c.u_lat}, method="nearest").drop_vars(c.lat_name)
            u60_name = f"u_{c.u_lat:.0f}N"

        # robust vortex mean wind (area-mean of zonal-mean U over 55–75N)
        u_vortex = self._area_mean_latlon(ua_zm, c.vortex_lat_band)

        # polar cap diagnostics (60–90N)
        zpc = self._area_mean_latlon(zg_zm, c.polar_cap)
        tpc = self._area_mean_latlon(ta_zm, c.polar_cap)

        ds = xr.Dataset(
            data_vars={
                u60_name: u60,
                "u_vortex": u_vortex,
                "z_pcap": zpc,
                "t_pcap": tpc,
            },
            coords={
                c.time_name: ua[c.time_name],
                c.p_name: ua[c.p_name],
            },
            attrs={
                "name": name,
                "polar_cap": f"{c.polar_cap[0]}-{c.polar_cap[1]}N",
                "u_lat_band": "" if c.u_lat_band is None else f"{c.u_lat_band[0]}-{c.u_lat_band[1]}N",
                "vortex_lat_band": f"{c.vortex_lat_band[0]}-{c.vortex_lat_band[1]}N",
            },
        )
        return ds

    def compare_to_ref(
        self,
        ds_exp: xr.Dataset,
        ds_ref: xr.Dataset,
        label: str = "vs_ref",
    ) -> xr.Dataset:
        """
        Adds bias time series and RMSE summaries. Assumes time coords align (same months).
        """
        c = self.cfg
        # align time safely
        exp, ref = xr.align(ds_exp, ds_ref, join="inner")

        out = exp.copy()

        # bias series for every shared variable
        common_vars = [v for v in exp.data_vars if v in ref.data_vars]
        for v in common_vars:
            out[f"{v}_bias_{label}"] = exp[v] - ref[v]

        # RMSE summaries (all months + winter months)
        months = self._month_numbers(out[c.time_name])
        winter_mask = xr.DataArray(
            np.isin(months.values, np.array(c.winter_months, dtype=int)),
            dims=(c.time_name,),
            coords={c.time_name: out[c.time_name]},
        )

        for v in common_vars:
            err2 = (exp[v] - ref[v]) ** 2
            out[f"{v}_rmse_all_{label}"] = np.sqrt(err2.mean(c.time_name))
            # winter RMSE (DJF)
            err2_w = err2.where(winter_mask, drop=True)
            if err2_w.sizes.get(c.time_name, 0) > 0:
                out[f"{v}_rmse_winter_{label}"] = np.sqrt(err2_w.mean(c.time_name))
            else:
                out[f"{v}_rmse_winter_{label}"] = xr.full_like(out[f"{v}_rmse_all_{label}"], np.nan)

        return out

    def save(self, ds: xr.Dataset, out_nc: str | Path):
        out_nc = Path(out_nc)
        out_nc.parent.mkdir(parents=True, exist_ok=True)
        ds.to_netcdf(out_nc)
        return out_nc


In [3]:
if __name__ == "__main__":
    diag_dir = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper"
    out_dir = Path(diag_dir) / "figure_data" / "ssw_monthly"
    out_dir.mkdir(parents=True, exist_ok=True)

    plev = np.array(
        [100000, 92500, 85000, 70000, 60000, 50000, 40000, 30000, 25000, 20000,
         15000, 10000, 7000, 5000, 3000, 2000, 1000, 500, 100],
        dtype=np.float64
    )

    cfg = SswMonthlyConfig(
        time_name="time", p_name="plev", lat_name="lat", lon_name="lon",
        ua_var="ua", va_var="va", ta_var="ta", zg_var="zg",
        target_plev_pa=plev,
        polar_cap=(60.0, 90.0),
        u_lat_band=(55.0, 65.0),
        vortex_lat_band=(55.0, 75.0),
        winter_months=(12, 1, 2),
    )
    analyzer = SswMonthlyAnalyzer(cfg)

    ref_dict = {
        "name": "ERA5",
        "path": "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/E3SMv3_testings/ERA5",
        "period": "2012-2013",
    }
    ref_key = ref_dict["name"]
    
    # --- build ERA5 reference indices once ---
    da_ref_u, da_ref_t, da_ref_z = load_era5_for_ssw(ref_dict)
    ds_ref = analyzer.compute_indices(da_ref_u, da_ref_t, da_ref_z, name=ref_key)

    exp_keys = ["CTRL", "DARTOP", "ERA5OP", "ERA5DL", "ERA5BL", "DARTBL", "DARTBA", "DARTAE"]

    for exp_key in exp_keys:
        da_exp_u = load_eam_cmip6_monthly(exp_key, "ua")
        da_exp_t = load_eam_cmip6_monthly(exp_key, "ta")
        da_exp_z = load_eam_cmip6_monthly(exp_key, "zg")

        ds_exp = analyzer.compute_indices(da_exp_u, da_exp_t, da_exp_z, name=exp_key)
        ds_vs  = analyzer.compare_to_ref(ds_exp, ds_ref, label=ref_key)

        out_nc = out_dir / f"ssw_monthly_{exp_key}_vs_{ref_key}.nc"
        analyzer.save(ds_vs, out_nc)
        print("Wrote:", out_nc)


Wrote: /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/ssw_monthly/ssw_monthly_CTRL_vs_ERA5.nc
Wrote: /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/ssw_monthly/ssw_monthly_DARTOP_vs_ERA5.nc
Wrote: /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/ssw_monthly/ssw_monthly_ERA5OP_vs_ERA5.nc
Wrote: /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/ssw_monthly/ssw_monthly_ERA5DL_vs_ERA5.nc
Wrote: /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/ssw_monthly/ssw_monthly_ERA5BL_vs_ERA5.nc
Wrote: /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/ssw_monthly/ssw_monthly_DARTBL_vs_ERA5.nc
Wrote: /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/ssw_monthly/ssw_monthly_DARTBA_vs_ERA5.nc
Wrote: /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/ssw_monthly/ssw_mont

In [ ]:
if __name__ == "__main__":
    diag_dir = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper"
    out_dir = Path(diag_dir) / "figure_data" / "ssw_monthly"
    out_dir.mkdir(parents=True, exist_ok=True)

    plev = np.array(
        [100000, 92500, 85000, 70000, 60000, 50000, 40000, 30000, 25000, 20000,
         15000, 10000, 7000, 5000, 3000, 2000, 1000, 500, 100],
        dtype=np.float64
    )

    cfg = SswMonthlyConfig(
        time_name="time", p_name="plev", lat_name="lat", lon_name="lon",
        ua_var="ua", va_var="va", ta_var="ta", zg_var="zg",
        target_plev_pa=plev,
        polar_cap=(60.0, 90.0),
        u_lat_band=(55.0, 65.0),
        vortex_lat_band=(55.0, 75.0),
        winter_months=(12, 1, 2),
    )
    analyzer = SswMonthlyAnalyzer(cfg)

    ref_dict = {
        "name": "ERAI",
        "path": "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/E3SMv3_testings/ERAI",
        "period": "2012-2013",
    }
    ref_key = ref_dict["name"]
    
    # --- build ERA5 reference indices once ---
    da_ref_u, da_ref_t, da_ref_z = load_era5_for_ssw(ref_dict)
    ds_ref = analyzer.compute_indices(da_ref_u, da_ref_t, da_ref_z, name=ref_key)

    exp_keys = ["CTRL", "DARTOP", "ERA5OP", "ERA5DL", "ERA5BL", "DARTBL", "DARTBA", "DARTAE"]

    for exp_key in exp_keys:
        da_exp_u = load_eam_cmip6_monthly(exp_key, "ua")
        da_exp_t = load_eam_cmip6_monthly(exp_key, "ta")
        da_exp_z = load_eam_cmip6_monthly(exp_key, "zg")

        ds_exp = analyzer.compute_indices(da_exp_u, da_exp_t, da_exp_z, name=exp_key)
        ds_vs  = analyzer.compare_to_ref(ds_exp, ds_ref, label=ref_key)

        out_nc = out_dir / f"ssw_monthly_{exp_key}_vs_{ref_key}.nc"
        analyzer.save(ds_vs, out_nc)
        print("Wrote:", out_nc)
